# Lab type: review
# Course: ML302 — Transformer Models & Fine-Tuning
# Lesson: Debugging Fine-tuning Runs
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless rendering
import matplotlib.pyplot as plt

np.random.seed(42)
print('NumPy version:', np.__version__)

## Part 1: Loss Curve Pathology Identification

The cell below simulates four training runs and plots their loss curves. Match each curve to its pathology, then answer the judgment questions.

In [ ]:
def simulate_run(pattern, epochs=20):
    """Generate synthetic train/val loss curves for four pathologies."""
    x = np.linspace(0, 1, epochs)
    if pattern == 'healthy':
        train = 2.5 * np.exp(-3.5 * x) + 0.25 + 0.03 * np.random.randn(epochs)
        val   = 2.5 * np.exp(-3.0 * x) + 0.35 + 0.04 * np.random.randn(epochs)
    elif pattern == 'overfitting':
        train = 2.5 * np.exp(-4.5 * x) + 0.05 + 0.02 * np.random.randn(epochs)
        val   = 2.5 * np.exp(-2.0 * x) + 0.5 + 0.5 * x + 0.05 * np.random.randn(epochs)
    elif pattern == 'divergence':
        train = 0.5 + 2.0 * x + 0.3 * np.random.randn(epochs)
        val   = 0.6 + 2.2 * x + 0.35 * np.random.randn(epochs)
    elif pattern == 'plateau':
        train = 2.5 * np.exp(-1.5 * x) + 1.0 + 0.03 * np.random.randn(epochs)
        val   = 2.5 * np.exp(-1.3 * x) + 1.1 + 0.04 * np.random.randn(epochs)
    return np.clip(train, 0.01, None), np.clip(val, 0.01, None)

patterns = ['healthy', 'overfitting', 'divergence', 'plateau']
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, pattern in zip(axes.flat, patterns):
    train, val = simulate_run(pattern)
    ax.plot(train, label='Train loss', color='#50fa7b')
    ax.plot(val,   label='Val loss',   color='#ffb86c')
    ax.set_title(pattern.capitalize())
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
fig.tight_layout()
plt.savefig('/tmp/loss_curves.png', dpi=100)
print('Loss curves saved to /tmp/loss_curves.png')

# Print final epoch stats
print()
for pattern in patterns:
    train, val = simulate_run(pattern)
    print(f'{pattern:<12} — final train: {train[-1]:.3f}, final val: {val[-1]:.3f}, gap: {val[-1]-train[-1]:+.3f}')

**Question 1:** Match each pathology to its primary intervention from the table below. For each one, explain *why* that specific intervention addresses the root cause (not just the symptom).

| Pathology | Primary intervention |
|-----------|---------------------|
| Overfitting | ? |
| Divergence | ? |
| Plateau | ? |

Possible interventions (each used once): `Reduce learning rate by 5–10×`, `Add weight decay / switch to PEFT`, `Check whether target layers are actually trainable (gradient norms)`.

*(Write your matched pairs and explanations here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Overfitting → Add weight decay / switch to PEFT.** Overfitting means the model's effective parameter count is too large relative to the training set — it memorises instead of generalising. Weight decay penalises large weights, implicitly constraining the hypothesis space. PEFT (e.g., LoRA) goes further by freezing most weights, so the adapter can only learn a low-rank update. Both reduce the number of degrees of freedom available to overfit.

**Divergence → Reduce learning rate by 5–10×.** Divergence (loss spikes upward then fails to recover) is caused by gradient updates that overshoot the loss basin — the parameter changes are so large that the model moves away from good solutions on every step. Reducing the LR by 5–10× shrinks the step size, ensuring updates stay within a region where the loss landscape approximates a smooth bowl.

**Plateau → Check whether target layers are actually trainable (gradient norms).** A plateau where *both* train and val loss stall early is the signature of a training-loop bug — commonly, the layers that need to update aren't receiving gradients (frozen-layer bug, wrong `.requires_grad` setting). Reducing the LR or adding regularisation cannot fix zero gradients. Checking gradient norms per layer immediately reveals which layers are dead, making the root cause actionable.

</details>

**Question 2:** The 'plateau' curve shows both train and val loss decreasing initially then flattening at a loss of ~1.0 for 15 epochs. Two different root causes can produce this shape: (a) learning rate too low, or (b) a frozen-layer bug where a subset of layers aren't receiving gradients. Describe one concrete diagnostic step you would take to distinguish between these two causes before adjusting any hyperparameter.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Concrete diagnostic step:** Log gradient norms per layer (or per layer group) for one epoch without changing any hyperparameter. If the plateau is caused by a frozen-layer bug, you will see near-zero gradient norms for the affected layers throughout training — no signal is flowing through them regardless of learning rate. If the plateau is caused by an LR too low, gradient norms will be consistently positive (non-zero) across all trainable layers, indicating the model *is* receiving gradient signal but is updating too slowly.

The two causes produce opposite gradient-norm signatures: frozen bug → silent zero norms; LR too low → active but small norms. No hyperparameter change is needed to diagnose this — it is a measurement step.

</details>

## Part 2: Gradient Norm Interpretation

The Trainer logs gradient norms when `max_grad_norm` is set. The cell below simulates three training runs with different gradient norm signatures and prints their step-by-step norms.

In [ ]:
def simulate_grad_norms(scenario, steps=30):
    """
    Simulate logged pre-clipping gradient norms for three scenarios.
    clip_threshold = 1.0
    """
    clip = 1.0
    if scenario == 'healthy':
        # Norms start moderate, decrease as training stabilises
        raw = 0.8 * np.exp(-0.05 * np.arange(steps)) + 0.1 * np.random.rand(steps)
    elif scenario == 'lr_too_high':
        # Norms consistently exceed clip threshold — every step is clipped
        raw = 1.5 + 0.5 * np.random.rand(steps)
    elif scenario == 'frozen_layer_bug':
        # Norms are near-zero throughout — most layers aren't updating
        raw = 0.002 + 0.001 * np.random.rand(steps)
    return raw

scenarios = ['healthy', 'lr_too_high', 'frozen_layer_bug']
print(f'{'Step':>5}  ' + '  '.join(f'{s:<22}' for s in scenarios))
print('-' * 80)
for step in [0, 5, 10, 15, 20, 25, 29]:
    row = f'{step:>5}  '
    for s in scenarios:
        norms = simulate_grad_norms(s)
        clipped = min(norms[step], 1.0)
        flag = ' ← CLIPPED' if norms[step] >= 1.0 else ''
        row += f'{norms[step]:.4f} (eff: {clipped:.4f}){flag:<12}  '
    print(row)

print()
print('Clip threshold: 1.0')
print('"eff" = effective norm after clipping (what actually updates weights)')

**Question 3:** In the `lr_too_high` scenario, the raw gradient norm is consistently ~1.5–2.0, and every step is clipped to 1.0. Your learning rate is set to `2e-5`. A colleague says: 'The clipping protects us — the model is still updating correctly.' Explain why this reasoning is wrong by describing what constant clipping does to the *effective* learning rate compared to the *configured* learning rate.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Why constant clipping is not protective:** Gradient clipping rescales the gradient vector so its norm equals `max_grad_norm` when the raw norm exceeds the threshold. If the raw norm is 1.8 and the clip value is 1.0, the effective gradient is scaled down by a factor of `1.0/1.8 ≈ 0.56`. The parameter update is therefore `lr × (clipped_grad) = 2e-5 × (scaled_grad)` — the model is effectively training at a lower learning rate than configured, but *inconsistently*: steps where the norm is below 1.0 apply the full `2e-5`, while steps above clip apply a smaller effective rate. The model receives unpredictably scaled updates rather than the steady gradient signal needed for stable convergence.

**The correct diagnosis:** Consistent clipping at every step means the configured LR is too high for the current model-data regime. The fix is to reduce the LR (typically 5–10×) until most steps pass through without clipping, so the scheduled rate is actually the rate the model trains at.

</details>

**Question 4:** In the `frozen_layer_bug` scenario, gradient norms are near-zero throughout training. Training loss still decreases slightly. How is it possible for loss to decrease if most layers have near-zero gradients? What part of the model is actually updating, and how would you confirm this?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q4</summary>

**How loss can still decrease:** The output head (classification layer) and any truly trainable parameters — typically a linear head added on top of the frozen base — do receive gradients and update normally. Even a single linear layer can reduce loss on a simple dataset by learning a better mapping from frozen representations to labels, giving the appearance of a functioning training run.

**What is actually updating:** The final classification head (or the LoRA adapters if misconfigured to be partially trainable). The attention/transformer layers the user intended to adapt are receiving no gradient and remain at their pretrained values.

**How to confirm:** Inspect `requires_grad` and logged gradient norms per named parameter:
```python
for name, param in model.named_parameters():
    print(name, param.requires_grad, param.grad.norm() if param.grad is not None else "no grad")
```
Layers with `requires_grad=True` but `grad is None` (or norm ≈ 0) after a backward pass confirm the frozen-layer bug. Cross-reference with which modules you intended to fine-tune.

</details>

## Part 3: Evaluation Integrity

Two evaluation pitfalls are demonstrated below: evaluation set leakage detection and format-only evaluation.

In [ ]:
import hashlib

def hash_example(text):
    return hashlib.md5(text.encode()).hexdigest()

# Simulate a train/eval split where 3 examples accidentally appear in both sets
train_texts = [
    'Quarterly earnings beat analyst expectations by a wide margin.',
    'The new model achieves state-of-the-art results on three benchmarks.',
    'Scientists discover a new mechanism for photosynthesis in deep-sea algae.',
    'Premier League clubs announce record transfer window spending.',
    'Central bank holds interest rates steady amid inflation concerns.',
]

eval_texts = [
    'Scientists discover a new mechanism for photosynthesis in deep-sea algae.',  # leaked!
    'Quarterly earnings beat analyst expectations by a wide margin.',              # leaked!
    'The team won the championship after a dramatic penalty shootout.',
    'Premier League clubs announce record transfer window spending.',               # leaked!
    'Clinical trial results show 78% efficacy for the new treatment.',
]

train_hashes = set(hash_example(t) for t in train_texts)
eval_hashes  = set(hash_example(t) for t in eval_texts)
leaked = train_hashes & eval_hashes

print(f'Train size: {len(train_texts)}')
print(f'Eval size:  {len(eval_texts)}')
print(f'Leaked examples: {len(leaked)}')
print(f'Eval contamination rate: {len(leaked)/len(eval_texts)*100:.0f}%')
print()
for t in eval_texts:
    h = hash_example(t)
    status = 'LEAKED' if h in train_hashes else 'clean'
    print(f'  [{status}] {t[:70]}')

**Question 5:** The contamination check above uses exact-match hashing. In practice, fine-tuning datasets collected from the web may contain near-duplicate examples — the same content with minor rephrasing. Would exact-match hashing catch a reworded duplicate? What method does the lesson recommend for more thorough decontamination, and what threshold is considered standard in major LLM evaluations?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q5</summary>

**Exact-match hashing and near-duplicates:** No — exact-match hashing will not catch a reworded duplicate. If even one word is changed, the MD5 hash is entirely different. A rephrased version of the same benchmark question would pass the exact-match check undetected.

**Recommended method:** The lesson recommends n-gram overlap decontamination — typically 13-gram Jaccard or Bloom-filter-based substring matching. This catches near-duplicates where most of the text is shared even if individual words differ.

**Standard threshold in major LLM evaluations:** A 13-gram overlap of ≥ 80% is the threshold used by GPT-3, PaLM, and similar evaluations to flag and remove near-duplicate training examples from benchmark test sets. Any training example exceeding this threshold against a benchmark example is excluded before reporting results.

</details>

In [ ]:
# Format-only evaluation — demonstrates the blind spot
import json

def format_eval(model_output, expected_answer):
    """Evaluates only whether the output is valid JSON with required keys."""
    try:
        parsed = json.loads(model_output)
        if 'answer' in parsed and 'confidence' in parsed:
            return True, 'format_pass'
    except json.JSONDecodeError:
        return False, 'invalid_json'
    return False, 'missing_keys'

# Four model outputs: correct format but wrong content, and vice versa
test_cases = [
    ('{"answer": "Paris",        "confidence": 0.98}', 'Paris',   'correct + correct format'),
    ('{"answer": "London",       "confidence": 0.91}', 'Paris',   'wrong answer + correct format'),
    ('{"answer": "Berlin",       "confidence": 0.85}', 'Paris',   'wrong answer + correct format'),
    ('The capital of France is Paris.',                  'Paris',   'correct answer + wrong format'),
]

print(f'{'Output':<50} {'Expected':<10} {'Format eval':>12} {'Content correct?':>17}')
print('-' * 92)
for output, expected, description in test_cases:
    passed, reason = format_eval(output, expected)
    try:
        content_correct = json.loads(output).get('answer') == expected
    except Exception:
        content_correct = False
    print(f'{output[:48]:<50} {expected:<10} {str(passed):>12} {str(content_correct):>17}')

**Question 6:** The format-only evaluation passes the 'wrong answer + correct format' cases. A fine-tuning run trained on 800 examples achieves 96% format-eval accuracy after 5 epochs. The training set has a very consistent JSON template. Give two observable signals that would suggest the model has learned the format but not the underlying task, and describe one additional evaluation step that would distinguish between these.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q6</summary>

**Observable signal 1 — near-perfect train accuracy with random-chance content accuracy:** If the model achieves 96% format accuracy but content accuracy (evaluated separately on held-out examples requiring real reasoning) is near the random baseline (e.g., 25% for 4-class), the model has learned the JSON template but not the task.

**Observable signal 2 — no improvement on out-of-distribution prompts:** If you rephrase the evaluation question significantly (different wording, different order of fields requested), format accuracy collapses — a model that has learned the task generalises across phrasings, while a format-memoriser cannot.

**Additional evaluation step:** Add a set of content-correct / content-incorrect probe examples where the format is always valid (to neutralise format as a signal) and measure exact-match or F1 on the answer field independently of the format fields. If `answer` field accuracy is at random chance while the overall format eval reads 96%, the model has learned format without task understanding.

</details>

## Summary

> **Final check — answer in one sentence each.**

1. The diagnostic difference between overfitting and plateau in a loss curve: 
2. Constant gradient clipping at every training step signals: 
3. The minimum decontamination check before reporting benchmark results: 
4. Format-only evaluation misses: 

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Overfitting vs plateau in a loss curve:** Overfitting shows train loss continuing to decrease while val loss diverges upward; a plateau shows both train and val loss stalling at the same level, which points to a training-loop bug (frozen layers) or an LR too low, not a capacity problem.

2. **Constant gradient clipping at every step signals:** The configured learning rate is too high — the raw gradient consistently exceeds the clip threshold, meaning the model trains at an unpredictably scaled effective rate rather than the scheduled one.

3. **Minimum decontamination check before reporting benchmark results:** Compute 13-gram Jaccard overlap between every training example and every benchmark test example, and remove training examples that exceed the 80% overlap threshold.

4. **Format-only evaluation misses:** Whether the model is producing correct *answers* — a model that has memorised the output template will score perfectly on format while returning wrong content, making the metric useless as a proxy for task capability.

</details>